## 1. Import thư viện

In [53]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier

## 2. Load train/test + drop cols

In [ ]:
train = pd.read_csv("data/raw/UNSW_NB15_training-set.csv")
test  = pd.read_csv("data/raw/UNSW_NB15_testing-set.csv")

for df in (train, test):
    for col in ["attack_cat", "id"]:
        if col in df.columns:
            print("Dropping", col)
            df.drop(columns=[col], inplace=True)

Dropping attack_cat
Dropping id
Dropping attack_cat
Dropping id


In [76]:
train.info()
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 175341 entries, 0 to 175340
Data columns (total 43 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   dur                175341 non-null  float64
 1   proto              175341 non-null  object 
 2   service            175341 non-null  object 
 3   state              175341 non-null  object 
 4   spkts              175341 non-null  int64  
 5   dpkts              175341 non-null  int64  
 6   sbytes             175341 non-null  int64  
 7   dbytes             175341 non-null  int64  
 8   rate               175341 non-null  float64
 9   sttl               175341 non-null  int64  
 10  dttl               175341 non-null  int64  
 11  sload              175341 non-null  float64
 12  dload              175341 non-null  float64
 13  sloss              175341 non-null  int64  
 14  dloss              175341 non-null  int64  
 15  sinpkt             175341 non-null  float64
 16  di

## 3. Split X/y

In [ ]:
X_train = train.drop(columns=["label"])
y_train = train["label"]

X_test = test.drop(columns=["label"])
y_test = test["label"]

## 4. Feature engineering

In [ ]:
keep_state = ["FIN","INT","CON","REQ","RST"]
keep_service = ["-","dns","http","smtp","ftp-data","ftp","ssh","pop3"]
keep_proto = ["tcp","udp","arp","ospf","igmp_icmp_rtp"]

for X in (X_train, X_test):
    X.loc[~X["state"].isin(keep_state), "state"] = "others"
    X.loc[~X["service"].isin(keep_service), "service"] = "others"
    X.loc[X["proto"].isin(["igmp","icmp","rtp"]), "proto"] = "igmp_icmp_rtp"
    X.loc[~X["proto"].isin(keep_proto), "proto"] = "others"

## 5. Categorical / Numeric

In [ ]:
cat_cols = [c for c in X_train.columns if X_train[c].dtype == "object"]
num_cols = [c for c in X_train.columns if c not in cat_cols]

## 6. Scale numetic (fit train)

In [ ]:
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols]  = scaler.transform(X_test[num_cols])

## 7. Label encode (fit train + test)

In [ ]:
for c in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([X_train[c], X_test[c]]).astype(str))
    X_train[c] = le.transform(X_train[c].astype(str))
    X_test[c]  = le.transform(X_test[c].astype(str))

print("Number of features:", X_train.shape[1])

Number of features: 42


## 8. Feature importance - Train

In [ ]:
importance_dict = {"feature": X_train.columns}

clf = RandomForestClassifier(random_state=1)
clf.fit(X_train, y_train)
importance_dict["train"] = clf.feature_importances_

## 9. Feature importance - Train 10-fold

In [ ]:
kf = StratifiedKFold(n_splits=10, shuffle=True, random_state=1)
fi_list = []

for tr_idx, _ in kf.split(X_train, y_train):
    clf = RandomForestClassifier(random_state=1)
    clf.fit(X_train.iloc[tr_idx], y_train.iloc[tr_idx])
    fi_list.append(clf.feature_importances_)

importance_dict["train_10_fold"] = np.mean(fi_list, axis=0)

## 10. Feature importance - Combined

In [ ]:
total_X = pd.concat([X_train, X_test], ignore_index=True)
total_y = pd.concat([y_train, y_test], ignore_index=True)

clf = RandomForestClassifier(random_state=1)
clf.fit(total_X, total_y)
importance_dict["combined"] = clf.feature_importances_


## 11. Feature importance - Combined 10-fold

In [ ]:
fi_list = []

for tr_idx, _ in kf.split(total_X, total_y):
    clf = RandomForestClassifier(random_state=1)
    clf.fit(total_X.iloc[tr_idx], total_y.iloc[tr_idx])
    fi_list.append(clf.feature_importances_)

importance_dict["combined_10_fold"] = np.mean(fi_list, axis=0)

## 12. Build importance_df

In [ ]:
importance_df = pd.DataFrame(importance_dict)

for c in importance_df.columns:
    if c != "feature":
        importance_df[c] = importance_df[c] * 100 / importance_df[c].sum()

importance_df["mean"] = importance_df[[c for c in importance_df.columns if c != "feature"]].mean(axis=1)
importance_df.sort_values('train_10_fold', ascending=False)


,feature,train,train_10_fold,combined,combined_10_fold,mean
9,sttl,16.255094,17.387081,12.195223,13.627806,14.866301
31,ct_state_ttl,8.961909,8.853936,7.887640,7.346594,8.262520
10,dttl,7.311621,5.026960,3.998724,4.346553,5.170964
27,dmean,4.991935,4.970893,3.748818,3.759341,4.367747
8,rate,4.253581,4.470505,4.683989,4.706418,4.528623
12,dload,4.567464,4.088457,3.752310,3.620697,4.007232
23,tcprtt,3.202787,3.917761,3.483411,3.659161,3.565780
25,ackdat,1.805762,3.703582,3.014354,2.891019,2.853679
11,sload,3.310175,3.428756,4.250654,4.070798,3.765096
6,sbytes,3.067725,3.272822,4.640932,4.747160,3.932160


## 13. Save

In [ ]:
importance_df.to_csv("data/preprocess/importance.csv", index=False)
print("Saved: importance.csv")

Saved: importance.csv
